In [38]:
import pandas as pd
from difflib import SequenceMatcher
from tqdm import tqdm

def clean_text(text):
    return str(text).lower().strip()

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def remove_similar(train_comments, test_comments, threshold=0.8):
    test_clean = [clean_text(t) for t in test_comments]
    
    filtered_train = []
    
    for train in tqdm(train_comments):
        train_c = clean_text(train)
        
        is_overlap = False
        
        for test in test_clean:
            if similarity(train_c, test) > threshold:
                is_overlap = True
                break
        
        if not is_overlap:
            filtered_train.append(train)
    
    return filtered_train

In [39]:
# Load datasets
train_df = pd.read_csv("dataset\youtube_comments_dataset_10k.csv")
test_df = pd.read_csv(r"C:\NLP_PROJ\dataset\youtube_comments_dataset.csv")

train_comments = train_df["comment"].tolist()
test_comments = test_df["comment"].tolist()

# Clean dataset
filtered_comments = remove_similar(train_comments, test_comments, threshold=0.8)

# Create new dataframe
cleaned_train_df = train_df[train_df["comment"].isin(filtered_comments)]

# Save new dataset
cleaned_train_df.to_csv("cleaned_train.csv", index=False)

print("Original:", len(train_df))
print("Cleaned:", len(cleaned_train_df))

100%|██████████| 9980/9980 [13:03<00:00, 12.73it/s]

Original: 9980
Cleaned: 8054


In [40]:
cleaned_train_df['sentiment'].value_counts()

sentiment
positive    2860
neutral     2665
negative    2529
Name: count, dtype: int64

In [41]:
cleaned_train_df['sentiment'] = cleaned_train_df['sentiment'].map({'negative': 0, 'neutral': 1, 'positive': 2})
cleaned_train_df.to_csv("cleaned_train.csv", index=False)

C:\Users\addys\AppData\Local\Temp\ipykernel_56288\1130559082.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_train_df['sentiment'] = cleaned_train_df['sentiment'].map({'negative': 0, 'neutral': 1, 'positive': 2})


In [46]:
cleaned_train_df[cleaned_train_df['augmentation_technique'] == 'register_analytical']['sentiment'].value_counts()

sentiment
2    53
1    49
0    43
Name: count, dtype: int64

In [52]:
import pandas as pd

# Load your 10k augmented dataset
df = pd.read_csv("cleaned_train.csv")

# 1. Define the 'Gold Standard' techniques (Keep 100%)
# These are highly realistic and essential for sentiment
core_keep = [
    'original', 
    'register_slang', 
    'intensity_scaling'
]

# 2. Define the 'Structural Diversity' techniques (Keep 50% sample)
# These add value but can be repetitive; we sample to reduce synthetic bias
diverse_sample = [
    'direct_address', 
    'perspective_shift', 
    'aspect_rotation',
    'comparative_framing'
]

# 3. Define the 'Garbage' techniques (Keep 0%)
# Formal/Analytical registers don't exist on YouTube; temporal can be confusing
to_drop = [
    'register_analytical', 
    'register_semiformal', 
    'temporal_framing'
]

# --- EXECUTION ---

# Create the core dataframe
df_core = df[df['augmentation_technique'].isin(core_keep)]

# Create the sampled dataframe
df_sampled = df[df['augmentation_technique'].isin(diverse_sample)].sample(frac=0.55, random_state=42)

# Combine them
df_filtered = pd.concat([df_core, df_sampled]).reset_index(drop=True)

# Final Stats
print(f"📊 Pruning Report:")
print(f"Original Count: {len(df)}")
print(f"Filtered Count: {len(df_filtered)}")
print(f"Reduction: {((len(df) - len(df_filtered)) / len(df)) * 100:.1f}%")

# Save for Fine-Tuning
df_filtered.to_csv("fine_tuning_dataset_final.csv", index=False)
print("\n✅ Your 'High-Quality' fine-tuning set is ready!")

📊 Pruning Report:
Original Count: 8054
Filtered Count: 6288
Reduction: 21.9%

✅ Your 'High-Quality' fine-tuning set is ready!


In [53]:
df_filtered.sentiment.value_counts()

sentiment
2    2233
1    2103
0    1952
Name: count, dtype: int64

In [54]:
# sample to 2058 samples per class
df_filtered = df_filtered.groupby('sentiment').apply(lambda x: x.sample(n=1950, random_state=42)).reset_index(drop=True)

C:\Users\addys\AppData\Local\Temp\ipykernel_56288\2385600104.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_filtered = df_filtered.groupby('sentiment').apply(lambda x: x.sample(n=1950, random_state=42)).reset_index(drop=True)


In [ ]:
df_filtered.to_csv("fine_tuning_dataset_fin al_balanced.csv", index=False)